# ISM 6564 — Practice Assignment 1

## Byte-Pair Encoding, by hand

**Week 1 — Text as Data & Tokenization · not marked, nothing to submit**

You are going to build BPE twice over: the **trainer** that learns a merge
list from a corpus, and the **encoder** that applies that merge list to any
word, including words the corpus never contained.

Then you will check your encoder against Hugging Face `tokenizers` — not
"looks about right", but an exact equality assertion on its merge list.

Work through steps 1–5 in order. Each has a check cell; if the assertion
passes, move on. Steps 6 and 7 are experiments to run and read.

This notebook runs top to bottom as shipped: unfinished steps print a `[ TODO ]`
reminder instead of raising.

---

## Setup

In [1]:
import random
import sys
from collections import Counter

import numpy as np
import torch

# Portable device selection: CUDA, else Apple Silicon MPS, else CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def todo(message):
    """Print a reminder instead of raising, so the notebook always runs."""
    print(f"[ TODO ]  {message}")


def ok(message):
    print(f"PASS  {message}")


print(f"python {sys.version.split()[0]}   torch {torch.__version__} ({device})")

python 3.12.13   torch 2.11.0 (mps)


### The corpus

Seven words with frequencies, from a retailer's support log. Small enough to
check by hand; rich enough that BPE will discover `retur`, `ing</w>`, and
`ed</w>` without ever being told that morphology exists.

`</w>` is the **end-of-word marker**. It is attached to the last character of
each word rather than standing alone — that is what Hugging Face
`tokenizers` does with `end_of_word_suffix`, and matching it now is what lets
step 6 compare the two exactly.

Without the marker, BPE cannot tell `ing` inside `singer` from `ing` at the
end of `shipping`, and detokenization has no way to know where to put the
spaces back.

In [2]:
EOW = "</w>"

CORPUS = {
    "refund": 7,
    "refunded": 3,
    "return": 11,
    "returned": 5,
    "returning": 2,
    "shipping": 13,
    "shipped": 6,
}

print(f"{sum(CORPUS.values())} word occurrences, {len(CORPUS)} distinct words")
print(f"{sum(len(w) * c for w, c in CORPUS.items())} characters")

47 word occurrences, 7 distinct words
336 characters


---

## Step 1 — Start from characters

Turn each word into a tuple of symbols: one per character, with `EOW` glued
onto the **last** character. `"refund"` becomes

```
('r', 'e', 'f', 'u', 'n', 'd</w>')
```

Return a dict mapping that tuple to the word's frequency. Tuples, not lists —
they need to be hashable, because they are dictionary keys and you will look
them up.

In [3]:
# TODO: implement init_splits.

def init_splits(freqs):
    """{word: count} -> {(symbol, symbol, ...): count} with EOW on the last char."""
    return None   # TODO

In [4]:
splits0 = init_splits(CORPUS)

EXPECTED_SPLITS0 = {
    ("r", "e", "f", "u", "n", "d</w>"): 7,
    ("r", "e", "f", "u", "n", "d", "e", "d</w>"): 3,
    ("r", "e", "t", "u", "r", "n</w>"): 11,
    ("r", "e", "t", "u", "r", "n", "e", "d</w>"): 5,
    ("r", "e", "t", "u", "r", "n", "i", "n", "g</w>"): 2,
    ("s", "h", "i", "p", "p", "i", "n", "g</w>"): 13,
    ("s", "h", "i", "p", "p", "e", "d</w>"): 6,
}

if splits0 is None:
    todo("Step 1 - implement init_splits()")
else:
    assert splits0 == EXPECTED_SPLITS0, f"got {splits0}"
    ok("step 1 - init_splits")
    for sym, c in splits0.items():
        print(f"  {c:>2}  {' '.join(sym)}")

[ TODO ]  Step 1 - implement init_splits()


---

## Step 2 — Count adjacent pairs

For every symbol sequence, count each **adjacent pair**, weighted by the
word's frequency. `("r", "e", "t", "u", "r", "n</w>")` with count 11
contributes 11 to each of `('r','e')`, `('e','t')`, `('t','u')`, `('u','r')`,
`('r','n</w>')`.

Note the pair `('r','e')` gets counted once here and once again from every
other word starting `re`. That accumulation across words is the entire idea:
BPE finds sequences that recur across the vocabulary, not within one word.

Return a `Counter`.

In [5]:
# TODO: implement pair_counts.
#
# Hint: zip(sym, sym[1:]) walks a sequence in adjacent pairs.

def pair_counts(splits):
    """{symbols: count} -> Counter of adjacent (left, right) symbol pairs."""
    return None   # TODO

In [6]:
pc0 = pair_counts(splits0) if splits0 is not None else None

if pc0 is None:
    todo("Step 2 - implement pair_counts()")
else:
    assert len(pc0) == 23, f"expected 23 distinct pairs, got {len(pc0)}"
    assert pc0[("r", "e")] == 28, pc0[("r", "e")]
    assert pc0[("s", "h")] == 19, pc0[("s", "h")]
    assert pc0[("e", "t")] == 18, pc0[("e", "t")]
    assert pc0[("i", "n")] == 15, pc0[("i", "n")]
    ok("step 2 - pair_counts")
    print("\n  top 8 pairs:")
    for pair, n in pc0.most_common(8):
        print(f"    {str(pair):<22} {n}")

[ TODO ]  Step 2 - implement pair_counts()


`('r','e')` wins with 28 — every word except the two `ship` words starts with
it. Below it sit four pairs tied on 19 and three tied on 18. Those ties are
not an artefact of a toy corpus; they happen constantly, and step 4 is where
you have to decide what to do about them.

---

## Step 3 — Apply one merge

Given the splits and a pair `(a, b)`, replace **every** adjacent occurrence of
`a` followed by `b` with the single symbol `a + b`.

Two things to get right:

- Scan left to right and skip two positions after a merge, so `('p','p')` in
  `p p p` merges once and leaves one `p`, rather than overlapping.
- Return a new dict. Mutating while iterating is how this goes wrong quietly.

In [7]:
# TODO: implement apply_merge.

def apply_merge(splits, pair):
    """Replace every adjacent occurrence of `pair` with the joined symbol."""
    return None   # TODO

In [8]:
splits1 = apply_merge(splits0, ("r", "e")) if splits0 is not None else None

EXPECTED_SPLITS1 = {
    ("re", "f", "u", "n", "d</w>"): 7,
    ("re", "f", "u", "n", "d", "e", "d</w>"): 3,
    ("re", "t", "u", "r", "n</w>"): 11,
    ("re", "t", "u", "r", "n", "e", "d</w>"): 5,
    ("re", "t", "u", "r", "n", "i", "n", "g</w>"): 2,
    ("s", "h", "i", "p", "p", "i", "n", "g</w>"): 13,
    ("s", "h", "i", "p", "p", "e", "d</w>"): 6,
}

if splits1 is None:
    todo("Step 3 - implement apply_merge()")
else:
    assert splits1 == EXPECTED_SPLITS1, f"got {splits1}"
    ok("step 3 - apply_merge")
    for sym, c in splits1.items():
        print(f"  {c:>2}  {' '.join(sym)}")

[ TODO ]  Step 3 - implement apply_merge()


---

## Step 4 — The training loop

Now repeat: count pairs, take the most frequent, merge it, record it. Twelve
times.

**The tie-break.** You saw four pairs tied on 19. `Counter.most_common(1)`
would return one of them, but *which* one depends on insertion order, which
depends on dictionary iteration order — reproducible today, not something to
rely on. So fix a rule and state it:

> Among all pairs with the maximum count, take the **lexicographically
> smallest** pair.

That is `min(...)` over the tied pairs. It is arbitrary. It is also written
down, which is the part that matters — step 6 shows you a library that made a
different arbitrary choice.

`learn_bpe` should return a list of records, one per merge:
`(pair, count, n_tied)`, where `n_tied` is how many pairs shared the winning
count. Keeping `n_tied` turns the tie-break from an invisible detail into
something you can see.

In [9]:
# TODO: implement best_pair and learn_bpe.

def best_pair(pc):
    """Most frequent pair; ties broken by taking the lexicographically smallest."""
    return None   # TODO


def learn_bpe(freqs, n_merges):
    """Return (merge_records, final_splits).

    merge_records is a list of (pair, count, n_tied), in learning order.
    """
    return None, None   # TODO

In [10]:
N_MERGES = 12
records, final_splits = learn_bpe(CORPUS, N_MERGES)

EXPECTED_MERGES = [
    ("r", "e"), ("h", "i"), ("hi", "p"), ("hip", "p"), ("s", "hipp"),
    ("re", "t"), ("ret", "u"), ("retu", "r"), ("i", "n"), ("in", "g</w>"),
    ("e", "d</w>"), ("shipp", "ing</w>"),
]

merges = [pair for pair, _, _ in records] if records is not None else None

if merges is None:
    todo("Step 4 - implement best_pair() and learn_bpe()")
else:
    assert merges == EXPECTED_MERGES, f"got {merges}"
    ok(f"step 4 - learn_bpe, {len(merges)} merges")
    print(f"\n  {'#':>2}  {'merge':<26} {'count':>5}  tied")
    for i, (pair, n, tied) in enumerate(records, 1):
        print(f"  {i:>2}  {str(pair):<26} {n:>5}  {tied}")
    print("\n  corpus after 12 merges:")
    for sym, c in final_splits.items():
        print(f"    {c:>2}  {' '.join(sym)}")

[ TODO ]  Step 4 - implement best_pair() and learn_bpe()


Look at what the merge list contains after twelve steps, none of it supplied
by you: `retur` (a stem), `ing</w>` and `ed</w>` (suffixes, and *only* in
word-final position because of the marker), and `shipping</w>` as one whole
token because it is frequent enough to be worth a vocabulary slot.

Also look at the `tied` column: 1, 4, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1. Six of
the twelve merges were decided by the tie-break rule rather than by frequency.

---

## Step 5 — The encoder

Training is done. Now use the merge list on a word — any word, including ones
the corpus never contained.

The rule: start from characters (with `EOW` on the last one), then walk the
merge list **in learning order**, applying each merge everywhere it fits.
Order is the whole algorithm — merge 3 can only fire on symbols that merges 1
and 2 created.

This is the same inner loop as `apply_merge`, on a single word instead of the
whole corpus.

In [11]:
# TODO: implement apply_merges (note the plural).

def apply_merges(word, merges):
    """Segment `word` by applying `merges` in order. Returns a list of symbols."""
    return None   # TODO

In [12]:
EXPECTED_SEGMENTS = {
    # seen in the training corpus
    "refund":     ["re", "f", "u", "n", "d</w>"],
    "returned":   ["retur", "n", "ed</w>"],
    "shipping":   ["shipping</w>"],
    "shipped":    ["shipp", "ed</w>"],
    # never seen -- and still segmented, with no [UNK] anywhere
    "refunding":  ["re", "f", "u", "n", "d", "ing</w>"],
    "reshipped":  ["re", "shipp", "ed</w>"],
    "unrefunded": ["u", "n", "re", "f", "u", "n", "d", "ed</w>"],
    "returns":    ["retur", "n", "s</w>"],
}

probe = apply_merges("refund", EXPECTED_MERGES)

if probe is None or merges is None:
    todo("Step 5 - implement apply_merges() (and finish step 4 first)")
else:
    for word, expected in EXPECTED_SEGMENTS.items():
        got = apply_merges(word, merges)
        assert got == expected, f"{word}: got {got}, expected {expected}"
    ok(f"step 5 - apply_merges on {len(EXPECTED_SEGMENTS)} words")
    print()
    for word, expected in EXPECTED_SEGMENTS.items():
        seen = "seen " if word in CORPUS else "UNSEEN"
        print(f"  {seen}  {word:<12} -> {expected}")

[ TODO ]  Step 5 - implement apply_merges() (and finish step 4 first)


`unrefunded` never appeared in training and still comes out as
`u | n | re | f | u | n | d | ed</w>`. Nothing was dropped and nothing became
`[UNK]`.

That is the answer to the open-vocabulary problem: **a subword vocabulary is
closed at the character level**, so anything built from known characters is
representable. Notice the cost, though — an unknown word costs eight tokens
where a known one costs one. Rare words are not impossible, they are
*expensive*. Keep that in mind when you price a corpus in Assignment 1.

---

## Step 6 — Check against a real implementation

Now train Hugging Face `tokenizers` on the same seven words with the same
end-of-word convention, and compare.

Two comparisons, and they give different answers:

1. **Merge lists** — these will *not* match, and that is fine.
2. **Your encoder against their encoder, on their merge list** — this must
   match exactly, and it is the real test of step 5.

Nothing to implement here. Run it and read it.

In [13]:
import json as _json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

hf_tok = Tokenizer(models.BPE(end_of_word_suffix=EOW))
hf_tok.pre_tokenizer = pre_tokenizers.Whitespace()
hf_trainer = trainers.BpeTrainer(
    vocab_size=1000, min_frequency=1, special_tokens=[],
    show_progress=False, end_of_word_suffix=EOW,
)

# The trainer wants text, not a frequency table, so write the words out.
training_text = " ".join(w for w, c in CORPUS.items() for _ in range(c))
hf_tok.train_from_iterator([training_text], hf_trainer)

hf_state = _json.loads(hf_tok.to_str())
hf_merges = [tuple(m) for m in hf_state["model"]["merges"]]

print(f"Hugging Face learned {len(hf_merges)} merges:")
for i, pair in enumerate(hf_merges[:12], 1):
    print(f"  {i:>2}  {pair}")

Hugging Face learned 23 merges:
   1  ('r', 'e')
   2  ('h', 'i')
   3  ('p', 'p')
   4  ('s', 'hi')
   5  ('shi', 'pp')
   6  ('t', 'u')
   7  ('re', 'tu')
   8  ('retu', 'r')
   9  ('i', 'n')
  10  ('in', 'g</w>')
  11  ('e', 'd</w>')
  12  ('shipp', 'ing</w>')


In [14]:
# Comparison 1: the merge lists.

if merges is None:
    todo("Step 6 - needs steps 1-4")
else:
    divergence = None
    for i, (mine, theirs) in enumerate(zip(merges, hf_merges)):
        if mine != theirs:
            divergence = i
            break

    print(f"{'#':>2}  {'yours':<26} {'hugging face':<26} same?")
    for i in range(min(len(merges), len(hf_merges), 12)):
        same = "yes" if merges[i] == hf_merges[i] else "NO"
        print(f"{i + 1:>2}  {str(merges[i]):<26} {str(hf_merges[i]):<26} {same}")

    if divergence is None:
        print("\nThe two merge lists agree on every position compared.")
    else:
        mine, theirs = merges[divergence], hf_merges[divergence]
        print(f"\nFirst divergence at merge {divergence + 1}: "
              f"you chose {mine}, Hugging Face chose {theirs}.")
        print(f"At that step the winning count was {records[divergence][1]} "
              f"and {records[divergence][2]} pairs were tied for it.")

[ TODO ]  Step 6 - needs steps 1-4


The lists diverge, and neither is wrong. Both implementations pick a pair with
the maximum count; they differ only in which of the tied pairs they take.
Yours takes the lexicographically smallest. The Rust implementation behind
`tokenizers` breaks ties on the internal token ids — effectively preferring
symbols that entered the vocabulary earlier.

**The practical consequence:** a merge list is not reproducible across
implementations, and neither is a vocabulary. This is one concrete reason a
model and its tokenizer travel together as a pair, and why "just retrain the
tokenizer" is never a free operation.

In [15]:
# Comparison 2: your ENCODER against theirs, on THEIR merge list.
#
# This one is not about tie-breaking. Given the same merges, the two encoders
# must produce the same segmentation, character for character.
#
# Every word below ends in d, n, or g. Step 7 explains why that matters.

SAFE_WORDS = ["refund", "refunded", "return", "returned", "returning",
              "shipping", "shipped", "refunding", "unrefunded",
              "reshipping", "funding", "spend", "hind"]

if probe is None or merges is None:
    todo("Step 6 - needs apply_merges() from step 5")
else:
    for word in SAFE_WORDS:
        mine = apply_merges(word, hf_merges)
        theirs = hf_tok.encode(word).tokens
        assert mine == theirs, f"{word}: yours {mine} != theirs {theirs}"
    ok(f"step 6 - encoder matches tokenizers on {len(SAFE_WORDS)} words")
    print()
    for word in SAFE_WORDS[:8]:
        print(f"  {word:<12} {apply_merges(word, hf_merges)}")

[ TODO ]  Step 6 - needs apply_merges() from step 5


---

## Step 7 — What a character alphabet loses

Your tokenizer's alphabet is not "all characters". It is "the characters that
appeared in the training corpus" — and, because of the end-of-word marker,
word-final characters are *separate symbols* from the same character
mid-word.

The seven training words end in only `d`, `n`, and `g`. So `d</w>`, `n</w>`,
and `g</w>` exist, and `p</w>` and `s</w>` do not.

Watch what a real tokenizer does when you hand it a word ending in `p`.

In [16]:
alphabet = sorted(t for t in hf_state["model"]["vocab"] if t.endswith(EOW) and len(t) <= 6)
print("word-final symbols in the learned alphabet:", alphabet)
print()

for word in ["ship", "refunds", "reship"]:
    theirs = hf_tok.encode(word).tokens
    joined = "".join(t.replace(EOW, "") for t in theirs)
    print(f"  {word:<9} -> {str(theirs):<28} reconstructs as {joined!r}"
          f"   {'LOSSLESS' if joined == word else 'CHARACTER LOST'}")

word-final symbols in the learned alphabet: ['d</w>', 'ed</w>', 'g</w>', 'n</w>']

  ship      -> ['shi']                      reconstructs as 'shi'   CHARACTER LOST
  refunds   -> ['refun', 'd']               reconstructs as 'refund'   CHARACTER LOST
  reship    -> ['re', 'shi']                reconstructs as 'reshi'   CHARACTER LOST


`ship` comes back as `shi`. The `p` was not replaced by an `[UNK]` and no
warning was raised — the symbol simply did not exist, so it was dropped.

This is the failure that byte-level BPE removes. Instead of an alphabet of
*characters seen in training*, GPT-2's tokenizer uses an alphabet of **all 256
byte values**. Every possible input is representable, so there is no path to
`[UNK]` and no path to a silently dropped character.

The cell below is the contrast, on the two tokenizers you will use in
Assignment 1.

In [17]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")   # WordPiece
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")                # byte-level BPE

TRICKY = ["Refund my order 😤 please", "café résumé", "日本語のテキスト", "🚀🛰️"]

print(f"{'string':<28} {'BERT [UNK]':>11}  {'GPT-2 round-trip':>17}")
for s in TRICKY:
    bert_pieces = bert_tok.tokenize(s)
    n_unk = bert_pieces.count(bert_tok.unk_token)
    roundtrip = gpt2_tok.decode(gpt2_tok.encode(s)) == s
    print(f"{s:<28} {n_unk:>11}  {str(roundtrip):>17}")

print()
for s in TRICKY:
    assert gpt2_tok.decode(gpt2_tok.encode(s)) == s, s
ok("step 7 - byte-level BPE round-trips every string above")

print()
print("BERT on the emoji string:", bert_tok.tokenize(TRICKY[0]))
print("BERT on 'café résumé'   :", bert_tok.tokenize("café résumé"),
      "  <- the accents are gone, and this is NOT reversible")

string                        BERT [UNK]   GPT-2 round-trip
Refund my order 😤 please               1               True
café résumé                            0               True
日本語のテキスト                               0               True
🚀🛰️                                    1               True

PASS  step 7 - byte-level BPE round-trips every string above

BERT on the emoji string: ['ref', '##und', 'my', 'order', '[UNK]', 'please']
BERT on 'café résumé'   : ['cafe', 'resume']   <- the accents are gone, and this is NOT reversible


Four failure modes, and it is worth keeping them apart:

| | What happens | Recoverable? |
|---|---|---|
| Symbol missing from a character alphabet | dropped silently | no |
| WordPiece meets an unrepresentable character | becomes `[UNK]` | no, but at least visible |
| WordPiece normalises `café` | becomes `cafe` | no — and it never looked like an error |
| Byte-level BPE meets anything | splits into more tokens | yes, exactly |

The third row is the nastiest, because nothing in the output says a decision
was made. Assignment 1 Task 4 asks you to write a normalization policy; that
row is what the policy is for.

---

## What to carry into the graded assignment

- **A subword vocabulary has a ceiling.** You saw why: after the merge budget
  is spent, the vocabulary stops growing no matter how much text arrives.
  Assignment 1 Task 2 measures that ceiling against a whitespace vocabulary
  that never stops growing.
- **Unseen words cost tokens, not correctness.** `unrefunded` took eight
  tokens instead of one. Assignment 1 Task 5 turns exactly that into dollars,
  and into a fairness question when the "rare" text is an entire language.
- **`[UNK]` is a property of the tokenizer, not of the text.** Byte-level BPE
  cannot produce one; WordPiece can. Assignment 1 Task 3 asks you to measure
  both and reconcile the results.
- **Tie-breaks, normalization, and end-of-word conventions are decisions.**
  They are written down somewhere in a config file, not derived from theory,
  and they are why a tokenizer and its model cannot be mixed and matched.

If you want to push further: rerun `learn_bpe` with 30 merges and watch
`refunded` and `returning` collapse into single tokens. The point at which a
word stops being decomposed is exactly the point at which the tokenizer has
decided it is common enough to memorise.